## Given a spike candidate, can the model classify it as a real pollution event or a fake sensor artifact?

1. Choose modelling features and remove leakage columns.
2. Split data by time, not randomly.
3. Train a simple baseline model.



- suspected_fake_sensor_artifact:            130
- confirmed_real_close_duration:              96

*for secondary experiment:*
- spatially_supported_different_duration:      4

*for semi-supervised:*
- unconfirmed_no_neighbours:                 404
- isolated_unconfirmed:                      387

Excluding the direct rule columns and use more general features, such as:

pm2p5,
rolling_mean,
rolling_std,
z_score,
pm2p5_diff,
target_duration_timesteps,
event_max_pm2p5,
event_mean_pm2p5,
drop_after_event,
hour_of_day,
day_of_week,
sensor reliability features,
recent missingness,
baseline variability

For a defensible first model, avoid these as input features:

rule_label,
ml_label,
suspected_fake,
sharp_jump,
sharp_drop_after_event,
returns_to_normal_quickly,
n_close_duration_neighbours,
close_duration_neighbour_ids,
duration_close_match

These define what is fake and what's real, so we avoid this so that our model doesn't just learn our rules but is able to seperate real from fake by itself. These columns either directly created the label or strongly reveal the rule used to create the label.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

labelled_spike_candidates = pd.read_csv("labelled_spike_candidates.csv")

# Make sure timestamps are datetime again
labelled_spike_candidates["timestamp"] = pd.to_datetime(
    labelled_spike_candidates["timestamp"]
)

print(labelled_spike_candidates.shape)
print(labelled_spike_candidates["rule_label"].value_counts())

(1021, 38)
rule_label
unconfirmed_no_neighbours                 404
isolated_unconfirmed                      387
suspected_fake_sensor_artifact            130
confirmed_real_close_duration              96
spatially_supported_different_duration      4
Name: count, dtype: int64


In [5]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Choose whether to include weak spatial labels
# --------------------------------------------------
# Recommended first run: False
# Later sensitivity test: True

include_weak_real_labels = False

if include_weak_real_labels:
    usable_labels = [
        "confirmed_real_close_duration",
        "spatially_supported_different_duration",
        "suspected_fake_sensor_artifact"
    ]
else:
    usable_labels = [
        "confirmed_real_close_duration",
        "suspected_fake_sensor_artifact"
    ]


# --------------------------------------------------
# 2. Keep only labelled examples for supervised learning
# --------------------------------------------------

model_df = labelled_spike_candidates[
    labelled_spike_candidates["rule_label"].isin(usable_labels)
].copy()


# --------------------------------------------------
# 3. Create binary ML target
# --------------------------------------------------
# real = 0
# fake_sensor = 1

model_df["target"] = np.select(
    [
        model_df["rule_label"].isin([
            "confirmed_real_close_duration",
            "spatially_supported_different_duration"
        ]),
        model_df["rule_label"] == "suspected_fake_sensor_artifact"
    ],
    [
        0,
        1
    ]
)

model_df["target_name"] = model_df["target"].map({
    0: "real",
    1: "fake"
})


# --------------------------------------------------
# 4. Check class balance
# --------------------------------------------------

print("Label counts:")
print(model_df["target_name"].value_counts())

print("\nRule-label counts:")
print(model_df["rule_label"].value_counts())

Label counts:
target_name
fake    130
real     96
Name: count, dtype: int64

Rule-label counts:
rule_label
suspected_fake_sensor_artifact    130
confirmed_real_close_duration      96
Name: count, dtype: int64


In [ ]:
# --------------------------------------------------
# Feature engineering
# --------------------------------------------------

model_df["timestamp"] = pd.to_datetime(model_df["timestamp"])

# Time features
model_df["hour"] = model_df["timestamp"].dt.hour
model_df["day_of_week"] = model_df["timestamp"].dt.dayofweek
model_df["month"] = model_df["timestamp"].dt.month
model_df["is_weekend"] = model_df["day_of_week"].isin([5, 6]).astype(int)

# Spike intensity relative to local baseline
model_df["pm2p5_above_baseline"] = (
    model_df["pm2p5"] - model_df["rolling_mean"]
)

model_df["pm2p5_ratio_to_baseline"] = (
    model_df["pm2p5"] / (model_df["rolling_mean"] + 1e-6)
)

model_df["event_peak_above_baseline"] = (
    model_df["event_max_pm2p5"] - model_df["rolling_mean"]
)

model_df["event_mean_above_baseline"] = (
    model_df["event_mean_pm2p5"] - model_df["rolling_mean"]
)

# rename for clarity
model_df = model_df.rename(columns={
    "n_neighbours_checked": "n_available_neighbours_checked"
})

candidate_features = [
    # Spike magnitude / local baseline
    "pm2p5",
    "rolling_mean",
    "rolling_std",
    "z_score",
    "pm2p5_diff",

    # Event-level shape
    "target_duration_timesteps",
    "event_max_pm2p5",
    "event_mean_pm2p5",

    # Engineered intensity features
    "pm2p5_above_baseline",
    "pm2p5_ratio_to_baseline",
    "event_peak_above_baseline",
    "event_mean_above_baseline",

    # Time context
    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    # spatial context
    "n_available_neighbours_checked",

    # Other sensor measurements if available
    "temperature",
    "humidity",
    "pm1p0",
    "pm10p0",
    "pm4p0",
    "voc",
    "nox",
    "co2"
]

# Keep only features that actually exist in your dataframe
feature_cols = [col for col in candidate_features if col in model_df.columns]

print("Features used:")
print(feature_cols)

print("\nNumber of features:", len(feature_cols))